In [ ]:
!pip install lightgbm
!pip install scikit-learn>=1.2.0
!pip install category_encoders>=2.5.0 

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
from operator import itemgetter
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import ast
import category_encoders as ce
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/My Drive/Colab Notebooks/data/all.csv')

print(df.head())


In [ ]:
def parse_techniques(x):
    try:
        return ast.literal_eval(x) if isinstance(x, str) else x
    except:
        return []

df['techniques'] = df['techniques'].apply(parse_techniques)

# Expand 'techniques' into separate binary columns
techniques_df = pd.DataFrame(df['techniques'].tolist(), index=df.index)
techniques_df.columns = [f"technique_{i}" for i in range(techniques_df.shape[1])]
df = pd.concat([df, techniques_df], axis=1)
df = df.drop('techniques', axis=1)

print("Data after preprocessing:")
df.head()


In [ ]:
df['RecipeId'] = df['RecipeId'].astype(int)
df['AuthorId'] = df['AuthorId'].astype(int)
df['Rating'] = df['Rating'].astype(float)
df = df.dropna()

In [ ]:
breakfast_recipe_ids = df.loc[df["IsBreakfast"] == 1, "RecipeId"].unique()
breakfast_recipe_ids_set = set(breakfast_recipe_ids)

lunch_recipe_ids = df.loc[df["IsLunch"] == 1, "RecipeId"].unique()
lunch_recipe_ids_set = set(lunch_recipe_ids)

snack_recipe_ids = df.loc[df["IsSnack"] == 1, "RecipeId"].unique()
snack_recipe_ids_set = set(snack_recipe_ids)

dinner_recipe_ids = df.loc[df["IsDinner"] == 1, "RecipeId"].unique()
dinner_recipe_ids_set = set(dinner_recipe_ids)

print(len(breakfast_recipe_ids_set))
print(len(lunch_recipe_ids_set))
print(len(snack_recipe_ids_set))
print(len(dinner_recipe_ids_set))

In [ ]:
feature_cols = ['AuthorId', 'RecipeId'] + [col for col in df.columns if col.startswith('technique_')]
target = 'Rating'
X = df[feature_cols]
y = df[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samplesK")
print(f"Testing set: {X_test.shape[0]} samples")

In [ ]:
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

In [ ]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_data_in_leaf': 20,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42
}

In [ ]:
gbm = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
     valid_sets=[train_data, test_data],
     callbacks=[lgb.early_stopping(stopping_rounds=5)]
)

In [ ]:
# 8. Evaluate the Model
y_pred = gbm.predict(X_test, num_iteration=gbm.best_iteration)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE: {rmse:.4f}")

# Feature Importance
lgb.plot_importance(gbm, max_num_features=20, importance_type='split')
plt.title('Feature Importances')
plt.show()

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
print(f"Validation MAE: {mae:.4f}")


In [ ]:
authors = defaultdict(int)
for author_id in X_test['AuthorId']:
  authors[author_id] += 1

X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

results_df = pd.DataFrame(columns=['AuthorId', 'RecipeId', 'TrueRating', 'PredictedRating'])

for index,row in X_test.iterrows():
  author_id = row['AuthorId']
  recipe_id = row['RecipeId']
  new_row = {'AuthorId': author_id, 'RecipeId': recipe_id, 'TrueRating': y_test[index], 'PredictedRating': y_pred[index]}
  results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

relevant_authors = {}

for author_id, author_data in authors.items():
    # Access the author ID and data
    if author_data > 2:
      relevant_authors[author_id] = author_data

In [ ]:
K = 10
precisions = []
for author_id, author_data in relevant_authors.items():
  selected_rows = results_df[(results_df['AuthorId'] == author_id) & (results_df['TrueRating'] > 4.0) & (results_df['RecipeId'].isin(dinner_recipe_ids_set))]
  if len(selected_rows) < k:
    continue

  k = math.ceil(min(K, len(selected_rows)))

  selected_rows_sorted = selected_rows.sort_values(by=['TrueRating'], ascending=False)
  top_k_rows = selected_rows_sorted.head(k)
  true_recipe_ids = top_k_rows['RecipeId'].tolist()

  selected_rows_sorted = selected_rows.sort_values(by=['PredictedRating'], ascending=False)
  top_k_rows = selected_rows_sorted.head(k)
  predicted_recipe_ids = top_k_rows['RecipeId'].tolist()

  n_rel = len(set(predicted_recipe_ids) & set(true_recipe_ids))
  precision = n_rel / k
  precisions.append(precision)

avg_precision = np.mean(precisions)
print(f"Average Precision@K for K={K}: {avg_precision:.4f}")



